# 02 — Historical Grain Analysis

This notebook validates the logical grain and investigates historical descriptive inconsistencies.

## Candidate logical grain

```text
ano_referencia
mes_referencia
codigo_fipe
ano_modelo
sigla_combustivel
```

The objective is to distinguish exact duplicates, non-exact grain collisions, and legitimate historical descriptive changes.


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

HISTORICAL_PATH = (
    PROJECT_ROOT / "data" / "bronze" / "historical" / "fipe_history_2026_08.parquet"
)
df_history = pd.read_parquet(HISTORICAL_PATH)
df_history.shape

In [ ]:
GRAIN_COLUMNS = [
    "ano_referencia",
    "mes_referencia",
    "codigo_fipe",
    "ano_modelo",
    "sigla_combustivel",
]

## Candidate-grain uniqueness


In [ ]:
grain_collision_mask = df_history.duplicated(subset=GRAIN_COLUMNS, keep=False)
grain_collision_rows = df_history[grain_collision_mask].copy()
len(grain_collision_rows)

In [ ]:
grain_group_sizes = (
    grain_collision_rows.groupby(GRAIN_COLUMNS, dropna=False)
    .size()
    .value_counts()
    .sort_index()
)
grain_group_sizes

In [ ]:
total_rows = len(df_history)
distinct_grain = df_history[GRAIN_COLUMNS].drop_duplicates().shape[0]
grain_excess_rows = total_rows - distinct_grain
total_rows, distinct_grain, grain_excess_rows

## Exact duplicates


In [ ]:
full_duplicate_mask = df_history.duplicated(keep=False)
full_duplicate_rows = df_history[full_duplicate_mask].copy()
len(full_duplicate_rows)

In [ ]:
exact_duplicate_excess = len(df_history) - len(df_history.drop_duplicates())
exact_duplicate_excess

Expected historical baseline:

- 156 rows participate in exact duplicate groups
- 83 rows are excess copies that can be removed deterministically


## Non-exact grain collisions


In [ ]:
deduplicated = df_history.drop_duplicates()
non_exact_collision_mask = deduplicated.duplicated(subset=GRAIN_COLUMNS, keep=False)
non_exact_collisions = deduplicated[non_exact_collision_mask].copy()
non_exact_collisions.shape

Expected baseline:

- 168 rows participate in non-exact grain collisions
- These require quarantine or targeted canonicalization


In [ ]:
non_grain_columns = [c for c in df_history.columns if c not in GRAIN_COLUMNS]
variation_summary = {}
for column in non_grain_columns:
    varying_groups = (
        non_exact_collisions.groupby(GRAIN_COLUMNS, dropna=False)[column]
        .nunique(dropna=False)
        .gt(1)
        .sum()
    )
    variation_summary[column] = int(varying_groups)
pd.Series(variation_summary).sort_values(ascending=False)

Historical result:

- `nome_marca`: 68 collision groups
- `nome_modelo`: 6 collision groups
- Other inspected non-grain attributes: no variation inside these collision groups


## Historical cardinality by FIPE code


In [ ]:
cardinality_by_fipe = df_history.groupby("codigo_fipe").agg(
    tipo_veiculo_nunique=("tipo_veiculo", "nunique"),
    nome_marca_nunique=("nome_marca", "nunique"),
    nome_modelo_nunique=("nome_modelo", "nunique"),
    nome_combustivel_nunique=("nome_combustivel", "nunique"),
    sigla_combustivel_nunique=("sigla_combustivel", "nunique"),
)
cardinality_by_fipe.describe()

In [ ]:
(cardinality_by_fipe > 1).sum()

Historical baseline by `codigo_fipe`:

- `tipo_veiculo`: 0 codes with more than one value
- `nome_marca`: 10
- `nome_modelo`: 455
- `nome_combustivel`: 186
- `sigla_combustivel`: 186

These counts show that descriptive values may evolve historically and therefore should not be added to the logical grain merely to force uniqueness.


## Brand variation examples


In [ ]:
brand_conflict_codes = cardinality_by_fipe[
    cardinality_by_fipe["nome_marca_nunique"] > 1
].index
brand_conflicts = (
    df_history[df_history["codigo_fipe"].isin(brand_conflict_codes)]
    .groupby(["codigo_fipe", "nome_marca"])
    .agg(
        occurrences=("codigo_fipe", "size"),
        first_year=("ano_referencia", "min"),
        last_year=("ano_referencia", "max"),
    )
    .reset_index()
)
brand_conflicts

## Model-name variation examples


In [ ]:
model_conflict_codes = cardinality_by_fipe[
    cardinality_by_fipe["nome_modelo_nunique"] > 1
].index
model_name_counts = (
    df_history[df_history["codigo_fipe"].isin(model_conflict_codes)]
    .groupby(["codigo_fipe", "nome_modelo"])
    .size()
    .rename("occurrences")
    .reset_index()
)
model_name_counts.sort_values(
    ["codigo_fipe", "occurrences"], ascending=[True, False]
).head(50)

## Fuel-history inspection


In [ ]:
fuel_conflict_codes = cardinality_by_fipe[
    cardinality_by_fipe["sigla_combustivel_nunique"] > 1
].index
fuel_history = (
    df_history[df_history["codigo_fipe"].isin(fuel_conflict_codes)]
    .groupby(["codigo_fipe", "sigla_combustivel", "nome_combustivel"])
    .agg(
        occurrences=("codigo_fipe", "size"),
        first_year=("ano_referencia", "min"),
        last_year=("ano_referencia", "max"),
    )
    .reset_index()
)
fuel_history.head(50)

## Grain conclusion

The candidate grain is retained:

```text
ano_referencia
mes_referencia
codigo_fipe
ano_modelo
sigla_combustivel
```

Operational treatment:

- Exact duplicate copies → deterministic deduplication
- Non-exact logical-grain collisions → quarantine
- Historical brand/model/fuel changes → preserve unless a separate high-confidence canonicalization rule exists

Production handling is implemented in `validate.py` and `transform.py`.
